# mx_dequantizer tests — Q10 (QDQ / fake-quant helper for QAT)

Scope: **only** algo-table item **Q10** — the `mx_dequantizer` round-trip used
to insert quantization noise for static-quantization evaluation / QAT.

Two bugs were fixed (algo table + review fallout):

1. **scale**: the shift was `shared_exp - mantissa_offset - calibrate_offset`.
   `mantissa_offset` is already inside `shared_exp`, so every value came out
   64x too small (2^-6).  Fix: `shift = shared_exp - calibrate_offset`.
2. **reshape**: for conv weights (`mx_weight_quantizer` flattens each kernel
   per OC), the trim used `original_shape[-1]` (= `Kw`) instead of `C*Kh*Kw`,
   so the final reshape crashed or scrambled the layout.

Round-trip invariant for a correct QDQ node (truncation, Q2): per block,
`|dequant(quant(y)) - y| <= 2^(shared_exp - calibrate_offset)` (one block-LSB).

Covers **activations** (intermediate + input paths) and **weights** (conv +
linear).  Semantics cross-verified against
`mxpt2e/weight_quant.mx_weight_dequantizer`.  Note: any QAT artifacts
produced through the old helper are stale by exactly 2^-6 -- regenerate.

Run all cells top to bottom; every assertion passing = tests pass.


In [ ]:
import os
import sys
# make npe_mx_emu importable whether jupyter runs from the repo root or from EL_nb/
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import torch

from npe_mx_emu import (
    base_mx_quantizer,
    input_mx_quantizer,
    mantissa_offset,
    mx_dequantizer,
    mx_weight_quantizer,
)

print("imports ok")


## Q10 — activations: intermediate and input paths

Both paths must recover the *original* tensor (the calibrate offset cancels
the upstream pre-scale): intermediate via `base_mx_quantizer`, input layer via
`input_mx_quantizer` (zero shared exponent).


In [ ]:

# ---- Q10 / Test A: intermediate activation, walkthrough vector ---------------
# (floor era, Q1: anchored cal is 8 -- floor(log2(0.4299)) = -2 -> c = 6 - (-2))
CAL = 8
x = torch.tensor([0.4299, 0.1382])
mx = base_mx_quantizer(x * 2.0 ** CAL, block_size=32, calibrate_offset=CAL)
# amax 110.05 -> E = floor(log2) = 6 -> shared_exp = E - m_off = 0 ; data = trunc = [110, 35]
assert mx.data.flatten().tolist()[:2] == [110, 35]
assert int(mx.shared_exp.flatten()[0]) == 0

deq = mx_dequantizer(mx)          # [110, 35] * 2^(0 - 8)  -> ORIGINAL units
want = torch.tensor([110 / 256.0, 35 / 256.0])   # [0.4296875, 0.13671875]
assert torch.allclose(deq[:2], want), f"Q10 A: {deq[:2].tolist()} != {want.tolist()}"
step = 2.0 ** (int(mx.shared_exp.flatten()[0]) - CAL)
assert (deq[:2] - x).abs().max() <= step, "Q10 A: error above one block-LSB"
print("Q10 A ok: dequant =", deq[:2].tolist(), f"(err {(deq[:2] - x).abs().max():.6f} <= step {step})")
print("floor recovered one bit: small element err is now 0.00148 (was 0.00538 under ceil)")


In [ ]:
# ---- Q10 / Test B: input-layer activation (zero shared exponent) -------------
CAL = 7
xi = torch.tensor([64.0, -32.0])       # integer sensor input, pre-scaled ints
mxi = input_mx_quantizer(xi, block_size=32, calibrate_offset=CAL)
assert int(mxi.shared_exp.flatten()[0]) == 0     # no exponent machinery

deq = mx_dequantizer(mxi)              # data * 2^(0 - CAL)
assert torch.allclose(deq[:2], torch.tensor([0.5, -0.25])), f"Q10 B: {deq[:2].tolist()}"
print("Q10 B ok: input path dequant =", deq[:2].tolist(), "= data * 2^-%d" % CAL)


## Q10 — weights: conv (kernel-flattened per OC) and linear

`mx_weight_quantizer` pre-scales each OC by `2^cal`, blocks the flattened
kernel, and stacks.  The dequantizer must reconstruct the original weight
shape (the old reshape crashed here) and meet the per-OC block-LSB bound.

Layout note (verbatim notebook quirk): **conv** weights loop per OC and stack
-> `data [OC, blocks, BS]`, `shared_exp [OC, blocks]`; **linear (2-D)** weights
take a single base call over the whole `(OC, C)` tensor and stack that one
result -> `data [1, OC, blocks, BS]`, `shared_exp [1, OC, blocks]`.  Blocking
is per-OC-row either way; only the indexing differs.


In [ ]:

# ---- Q10 / Test C: conv weights (4-D) -----------------------------------------
torch.manual_seed(11)
CAL = 9
w = torch.randn(2, 1, 2, 2)
for oc in range(w.shape[0]):
    w[oc] = w[oc] / w[oc].abs().max() * 0.8   # amax 0.8 -> scaled amax 0.8*2^9 >= 64,
#                                               guaranteed non-negative stored exponent

mxw = mx_weight_quantizer(w, block_size=32, calibrate_offset=CAL)
deq = mx_dequantizer(mxw)
assert deq.shape == w.shape, f"Q10 C: shape {tuple(deq.shape)} != {tuple(w.shape)}"
for oc in range(w.shape[0]):
    step = 2.0 ** (int(mxw.shared_exp[oc, 0]) - CAL)   # one block per OC here
    err = (deq[oc] - w[oc]).abs().max()
    assert err <= step, f"Q10 C: oc {oc} err {err} > step {step}"
    print(f"oc {oc}: stored exp {int(mxw.shared_exp[oc, 0])}, max err {err:.2e} <= step {step:.2e} | "
          f"rel err {((deq[oc] - w[oc]).abs().max() / w[oc].abs().max()).item():.4f}")
print("Q10 C ok: conv weights round-trip, original shape restored, per-OC bound met")


In [ ]:

# ---- Q10 / Test C2: linear weights (2-D) ---------------------------------------
torch.manual_seed(12)
CAL = 9
w2 = torch.randn(2, 3)
for oc in range(w2.shape[0]):
    w2[oc] = w2[oc] / w2[oc].abs().max() * 0.8   # same non-negative-exp guarantee

mxw2 = mx_weight_quantizer(w2, block_size=32, calibrate_offset=CAL)
# 2-D path layout: one stacked base call -> data [1, OC, blocks, BS],
# shared_exp [1, OC, blocks] (blocking is still per OC row)
print("2-D weight layout: data", tuple(mxw2.data.shape),
      "shared_exp", tuple(mxw2.shared_exp.shape))
deq2 = mx_dequantizer(mxw2)
assert deq2.shape == w2.shape, f"Q10 C2: shape {tuple(deq2.shape)} != {tuple(w2.shape)}"
for oc in range(w2.shape[0]):
    step = 2.0 ** (int(mxw2.shared_exp[0, oc, 0]) - CAL)
    err = (deq2[oc] - w2[oc]).abs().max()
    assert err <= step, f"Q10 C2: oc {oc} err {err} > step {step}"
    print(f"oc {oc}: stored exp {int(mxw2.shared_exp[0, oc, 0])}, max err {err:.2e} <= step {step:.2e}")
print("Q10 C2 ok: linear weights round-trip, trim parity with the old 2-D behavior")


## Q10 — randomized round-trip bound (activations 4-D and weights) + bug contrast

Per-element bound: an element's error is at most its own block's step
`2^(shared_exp - cal)` (truncation loses < 1 LSB of the mantissa).
The bug-contrast cell shows exactly what the old helper produced (x2^-6) --
any QAT artifact saved through it is stale and must be regenerated.


In [ ]:

# ---- Q10 / Test D: randomized bound, 4-D activations ----------------------------
torch.manual_seed(3)
CAL = 8
x = torch.rand(1, 2, 8, 8) * 0.5 + 0.25   # amax >= 0.25 -> scaled amax >= 64,
#                                           non-negative stored exponent by construction

mxa = base_mx_quantizer(x * 2.0 ** CAL, block_size=32, calibrate_offset=CAL)
deq = mx_dequantizer(mxa)
assert deq.shape == x.shape

steps = torch.exp2((mxa.shared_exp - CAL).unsqueeze(-1).float())  # [..., blocks, 1]
steps_full = steps.expand_as(mxa.data).reshape(*mxa.data.shape[:-2], -1)[..., : x.shape[-1]]
err = (deq - x).abs()
assert (err <= steps_full).all(), f"Q10 D: bound violated, max {err.max()}"
print(f"Q10 D ok: 4-D activation, max err {err.max():.3e} within per-block step")


In [ ]:

# ---- Q10 / Test D2: randomized bound, conv weights ------------------------------
torch.manual_seed(4)
CAL = 9
w = torch.randn(3, 2, 2, 2)
for oc in range(w.shape[0]):
    w[oc] = w[oc] / w[oc].abs().max() * 0.8   # per-OC amax 0.8, no wrap

mxw = mx_weight_quantizer(w, block_size=32, calibrate_offset=CAL)
deq = mx_dequantizer(mxw)
assert deq.shape == w.shape
for oc in range(w.shape[0]):
    step = 2.0 ** (int(mxw.shared_exp[oc, 0]) - CAL)
    err = (deq[oc] - w[oc]).abs().max()
    assert err <= step, f"Q10 D2: oc {oc} err {err} > step {step}"
print("Q10 D2 ok: conv weights randomized bound met for every OC")


In [ ]:

# ---- Q10 / Test E: bug contrast -- old helper was x2^-6 ------------------------
# SELF-CONTAINED: read cal/shape from the tensor, NOT from globals --
# Test D2 clobbers the global CAL (=9) after Test D built mxa with cal=8.
m_off = mantissa_offset("mx-e4bm8")
CAL_E = int(mxa.calibrate_offset)
old_shift = mxa.shared_exp - m_off - CAL_E                  # the old (buggy) term
old_deq = mxa.data * torch.exp2(old_shift.unsqueeze(-1).float())
old_deq = old_deq.reshape(*old_deq.shape[:-2], -1)[..., : mxa.original_shape[-1]].reshape(mxa.original_shape)
new_deq = mx_dequantizer(mxa)

mask = old_deq != 0
ratio = new_deq[mask] / old_deq[mask]
assert torch.allclose(ratio, torch.full_like(ratio, 2.0 ** m_off)), "Q10 E: ratio must be exactly 2^6"
print(f"Q10 E: old/new ratio == {float(ratio.flatten()[0]):.0f} everywhere (old helper was x2^-{m_off})")
print("       -> regenerate any QAT artifacts produced through the old mx_dequantizer")


In [ ]:
print("=" * 60)
print("ALL Q10 TESTS PASSED")
print("  scale: shift = shared_exp - calibrate_offset (2^-6 bug removed)")
print("  reshape: conv weights reconstruct original (OC, C, Kh, Kw) shape")
print("  round trip bound 2^(shared_exp - cal) holds for activations + weights")
